# Describing Algorithms as Surfaces of Success Rates

In [ ]:
import numpy as np
import pandas as pd
from lmfit import Model
from RamanModel import *

# distancias = np.linspace(0, 0.5, 0.05)
# intensidades = np.linspace(0.01, 1, 0.01)
# gamma_rel = np

# for distancia in distancias
#    for intensidade in intensidades
        

def generate_curves(num_samples, distancia, intensidade_relativa, ):
    center_range = (0.25, 0.75) # <-- ao inves disso
    amplitude_range = (0.01, 1)
    gamma_range = (0.001, 0.1)

    results_summary = []
    results_params = []

    x = np.linspace(0, 1, 1000)

    for curve_id in range(num_samples):
        y = np.zeros_like(x)
        true_params = []

        # --- Generate 2 peaks ---
        for i in range(2):
            center = np.random.uniform(*center_range)
            amplitude = np.random.uniform(*amplitude_range)
            gamma = np.random.uniform(*gamma_range)

            y += lorentzian(x, center, gamma, amplitude)

            true_params.append({
                "center": center,
                "gamma": gamma,
                "amplitude": amplitude,
            })

        # --- Sort peaks left → right ---
        true_params = sorted(true_params, key=lambda p: p["center"])

        # --- Build lmfit model ---
        model = Model(lorentzian, prefix='p1_') + Model(lorentzian, prefix='p2_')

        params = model.make_params()

        # Initial guesses (important for convergence)
        for i, p in enumerate(true_params):
            prefix = f"p{i+1}_"
            #params[f"{prefix}x0"].set(value=p["center"] * np.random.uniform(0.9, 1.1))
            #params[f"{prefix}gamma"].set(value=p["gamma"] * np.random.uniform(0.9, 1.1), min=1e-6)
            #params[f"{prefix}maximum"].set(value=p["amplitude"] * np.random.uniform(0.9, 1.1), min=0)

            params[f"{prefix}x0"].set(value=np.random.uniform(0, 1))
            params[f"{prefix}gamma"].set(value=np.random.uniform(0.001, 0.1), min=1e-6)
            params[f"{prefix}maximum"].set(value=np.random.uniform(0.01, 1), min=0)

        # --- Fit ---
        result = model.fit(y, params, x=x)

        # --- Extract fitted parameters ---
        fitted_params = []
        for i in range(2):
            prefix = f"p{i+1}_"
            fitted_params.append({
                "center": result.params[f"{prefix}x0"].value,
                "gamma": result.params[f"{prefix}gamma"].value,
                "amplitude": result.params[f"{prefix}maximum"].value,
            })

        # Sort fitted peaks too (important!)
        fitted_params = sorted(fitted_params, key=lambda p: p["center"])

        # --- Check relative error (≤5%) ---
        success = 1
        for t, f in zip(true_params, fitted_params):
            for key in ["center", "gamma", "amplitude"]:
                if t[key] == 0:
                    continue
                rel_error = abs((f[key] - t[key]) / t[key])
                if rel_error > 0.05:
                    success = 0
                    break
            if success == 0:
                break

        # --- Compute derived values ---
        amps = [p["amplitude"] for p in true_params]
        rel_intensity = min(amps) / max(amps)

        gamma1 = true_params[0]["gamma"]
        gamma2 = true_params[1]["gamma"]

        peak_distance = abs(true_params[1]["center"] - true_params[0]["center"])

        # --- Save summary row ---
        results_summary.append({
            "curve_id": curve_id,
            "relative_intensity": rel_intensity,
            "gamma_1": gamma1,
            "gamma_2": gamma2,
            "peak_distance": peak_distance,
            "fit_success": success
        })

        # --- Save detailed parameters ---
        for i, p in enumerate(true_params):
            results_params.append({
                "curve_id": curve_id,
                "peak_id": i,
                "center": p["center"],
                "gamma": p["gamma"],
                "amplitude": p["amplitude"]
            })

    # --- Write CSVs ---
    df_summary = pd.DataFrame(results_summary)
    df_params = pd.DataFrame(results_params)

    df_summary.to_csv("curve_summary.csv", index=False)
    df_params.to_csv("curve_parameters.csv", index=False)

    return df_summary, df_params
    

In [ ]:
df_summary, df_params = generate_curves(100000)

In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("curve_summary.csv")

df["dist_bin"] = pd.cut(df["peak_distance"], bins=10)
df["int_bin"] = pd.cut(df["relative_intensity"], bins=10)
df["gamma1_bin"] = pd.cut(df["gamma_1"], bins=10)
df["gamma2_bin"] = pd.cut(df["gamma_2"], bins=10)

dist_success = df.groupby(["dist_bin", "int_bin", "gamma1_bin", "gamma2_bin"])["fit_success"].agg(["mean", "count"])
dist_success = dist_success.rename(columns={"mean": "success_rate"})

safe_dist_bins = dist_success[dist_success["success_rate"] >= 0.50]

print(safe_dist_bins)

                                                                            success_rate  \
dist_bin            int_bin          gamma1_bin         gamma2_bin                         
(-0.000497, 0.0498] (0.00943, 0.109] (0.000902, 0.0109] (0.000901, 0.0109]      0.578947   
                                                        (0.0109, 0.0208]        0.631579   
                                                        (0.0208, 0.0307]        0.777778   
                                                        (0.0307, 0.0406]        0.578947   
                                                        (0.0406, 0.0505]        0.818182   
...                                                                                  ...   
(0.448, 0.498]      (0.901, 1.0]     (0.0901, 0.1]      (0.0109, 0.0208]        1.000000   
                                                        (0.0208, 0.0307]        1.000000   
                                                        (0.0307, 0.0406]        

In [5]:
%matplotlib notebook

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("curve_summary.csv")

# ============================================================
# PARAMETERS
# ============================================================

N_DISTANCE_BINS = 5

# bins for voxel probability volume
N_GAMMA1_BINS = 8
N_GAMMA2_BINS = 8
N_INTENSITY_BINS = 8

# ============================================================
# CREATE PEAK DISTANCE BINS
# ============================================================

distance_bins = np.linspace(
    df["peak_distance"].min(),
    df["peak_distance"].max(),
    N_DISTANCE_BINS + 1
)

df["distance_bin"] = pd.cut(
    df["peak_distance"],
    bins=distance_bins,
    include_lowest=True,
    labels=False
)

# ============================================================
# FUNCTION: RAW 3D SCATTER
# ============================================================

def plot_raw_scatter(subdf, bin_id, dmin, dmax):

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    success = subdf[subdf["fit_success"] == 1]
    failure = subdf[subdf["fit_success"] == 0]

    ax.scatter(
        success["gamma_1"],
        success["gamma_2"],
        success["relative_intensity"],
        c='green',
        s=20,
        alpha=0.7,
        label='Success'
    )

    ax.scatter(
        failure["gamma_1"],
        failure["gamma_2"],
        failure["relative_intensity"],
        c='red',
        s=20,
        alpha=0.7,
        label='Failure'
    )

    ax.set_xlabel("Gamma 1")
    ax.set_ylabel("Gamma 2")
    ax.set_zlabel("Relative Intensity")

    title = (
        f"Peak Distance Range: "
        f"{dmin:.4f} ≤ Δx < {dmax:.4f}"
    )
    ax.set_title(title)

    ax.legend()

    plt.tight_layout()
    plt.show()

# ============================================================
# FUNCTION: PROBABILITY VOXEL VOLUME
# ============================================================

def plot_probability_volume(subdf, bin_id, dmin, dmax):

    # --------------------------------------------------------
    # DEFINE BINS
    # --------------------------------------------------------

    g1_bins = np.linspace(
        df["gamma_1"].min(),
        df["gamma_1"].max(),
        N_GAMMA1_BINS + 1
    )

    g2_bins = np.linspace(
        df["gamma_2"].min(),
        df["gamma_2"].max(),
        N_GAMMA2_BINS + 1
    )

    ri_bins = np.linspace(
        df["relative_intensity"].min(),
        df["relative_intensity"].max(),
        N_INTENSITY_BINS + 1
    )

    # --------------------------------------------------------
    # ASSIGN BIN INDICES
    # --------------------------------------------------------

    subdf = subdf.copy()

    subdf["g1_bin"] = pd.cut(
        subdf["gamma_1"],
        bins=g1_bins,
        labels=False,
        include_lowest=True
    )

    subdf["g2_bin"] = pd.cut(
        subdf["gamma_2"],
        bins=g2_bins,
        labels=False,
        include_lowest=True
    )

    subdf["ri_bin"] = pd.cut(
        subdf["relative_intensity"],
        bins=ri_bins,
        labels=False,
        include_lowest=True
    )

    # --------------------------------------------------------
    # COMPUTE SUCCESS PROBABILITY PER VOXEL
    # --------------------------------------------------------

    grouped = subdf.groupby(
        ["g1_bin", "g2_bin", "ri_bin"]
    )["fit_success"]

    probability = grouped.mean()
    count = grouped.count()

    # --------------------------------------------------------
    # CREATE 3D ARRAYS
    # --------------------------------------------------------

    probs = np.zeros((
        N_GAMMA1_BINS,
        N_GAMMA2_BINS,
        N_INTENSITY_BINS
    ))

    filled = np.zeros_like(probs, dtype=bool)

    for (i, j, k), p in probability.items():

        if np.isnan(i) or np.isnan(j) or np.isnan(k):
            continue

        i = int(i)
        j = int(j)
        k = int(k)

        probs[i, j, k] = p
        filled[i, j, k] = True

    # --------------------------------------------------------
    # COLORS
    # --------------------------------------------------------

    cmap = cm.get_cmap("RdYlGn")

    norm = Normalize(vmin=0, vmax=1)

    colors = cmap(norm(probs))

    # transparency
    colors[..., -1] = filled * 0.8

    # --------------------------------------------------------
    # PLOT
    # --------------------------------------------------------

    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    ax.voxels(
        filled,
        facecolors=colors,
        edgecolor='k'
    )

    ax.set_xlabel("Gamma 1 Bin")
    ax.set_ylabel("Gamma 2 Bin")
    ax.set_zlabel("Rel. Intensity Bin")

    title = (
        f"Peak Distance Range: "
        f"{dmin:.4f} ≤ Δx < {dmax:.4f}"
    )
    ax.set_title(title)

    # --------------------------------------------------------
    # COLORBAR
    # --------------------------------------------------------

    mappable = cm.ScalarMappable(
        norm=norm,
        cmap=cmap
    )

    mappable.set_array([])

    cbar = plt.colorbar(
        mappable,
        ax=ax,
        shrink=0.7
    )

    cbar.set_label("Fit Success Probability")

    plt.tight_layout()
    plt.show()

# ============================================================
# MAIN LOOP OVER DISTANCE BINS
# ============================================================

for bin_id in range(N_DISTANCE_BINS):

    subdf = df[df["distance_bin"] == bin_id]

    dmin = distance_bins[bin_id]
    dmax = distance_bins[bin_id + 1]

    print(f"\nDistance Bin {bin_id}")
    print(f"Samples: {len(subdf)}")

    # --------------------------------------------------------
    # RAW SCATTER
    # --------------------------------------------------------

    plot_raw_scatter(subdf, bin_id, dmin, dmax)

    # --------------------------------------------------------
    # PROBABILITY VOLUME
    # --------------------------------------------------------

    plot_probability_volume(subdf, bin_id, dmin, dmax)

RuntimeError: 'widget' is not a recognised GUI loop or backend name

# Symbolic Regression

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("curve_summary.csv")

X = df[["relative_intensity", "gamma_1", "gamma_2", "peak_distance"]].values
y = df["fit_success"].values

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
from pysr import PySRRegressor

model = PySRRegressor(
    niterations=1000,

    # Keep it simple first
    binary_operators=["+", "-", "*", "/"],
    unary_operators=[],

    # Encourage simplicity
    model_selection="best",
    parsimony=1e-3,

    loss="loss(x, y) = (x - y)^2",

    # Limit complexity (VERY important)
    maxsize=20,
)

model.fit(X, y)

In [ ]:
print(model)